# 8. Triangle communities: derive and solve the cavity equations

A single triangle has degree two at every vertex, so it has an empty 3-core.
An extensive 3-core must instead come from triangles supporting one another.
We will describe that support, derive its probability distribution, and find
where a nonzero solution first appears.

The main prediction is a **discontinuous transition** near
$\lambda_c=1.22860$, with a positive right-hand density of about $0.08111$.
This is the triangle **cavity prediction**, whose assumptions we explain below.
It is not a theorem for our finite random graphs. Notebook 10 compares it with
saved measurements and with the configuration-model theory in notebook 05.

This notebook only imports dependencies and defines functions. Loading it with
`%run` does not simulate graphs, draw figures, or write files. Numerical examples
are shown as optional snippets; the calculations and diagnostic figures are
executed in notebook 10.

## The model and the right kind of tree

The sampler in notebook 02 first draws membership counts
$D_i\sim\operatorname{Poisson}(\lambda)$, conditions their total to be divisible
by three, and matches their membership slots uniformly into groups of three.
The three members of a group are joined. Our finite graph is simple: loops are
removed and repeated edges are merged. These coincidences disappear from the
neighbourhood of a uniformly chosen vertex in the sparse limit with fixed
positive $\lambda$.

Thus $\lambda$ is the limiting mean number of **triangle memberships**, whereas
the limiting graph degree is

$$
K=2D,\qquad D\sim\operatorname{Poisson}(\lambda),\qquad
\mathbb E[K]=2\lambda.
$$

The projected graph contains triangles even in its local limit. It is not an
ordinary tree. Instead, keep a second kind of vertex representing each group.
This **factor graph** is bipartite: individual vertices connect to group
vertices, and every group vertex has degree three. Its local limit is a tree
with triangles replacing group vertices upon projection.

Following an incidence reaches a size-biased membership count $D^*$, but the
number of *other* memberships is again Poisson:

$$
\mathbb P(D^*-1=j)
=\frac{(j+1)\mathbb P(D=j+1)}{\lambda}
=e^{-\lambda}\frac{\lambda^j}{j!}.
$$

Consequently the root and a vertex reached through one group have the same
Poisson number of available groups, after the incoming group is excluded.
This equality is special to the Poisson choice.

The local weak-limit framework is established in
[van der Hofstad, Komjáthy and Vadon, *Random intersection graphs with
communities*, Theorems 2.8 and 2.14, and Corollary 2.9](https://arxiv.org/pdf/1809.02514).
It explains why different branches of the limiting factor tree can be treated
independently. It does not make the two neighbours **within one triangle**
independent.

## What do the messages mean?

Core pruning repeatedly deletes a vertex with fewer than three remaining
neighbours. A cavity message asks a conditional question: **what support could
this branch provide if the vertex receiving it were kept present?** It is not
an unconditional declaration that every vertex on that branch survives.

For an individual $u$ incident to a triangle $a$, let

$$
S_{u\to a}=\sum_{b\ni u,\ b\ne a} M_{b\to u}
$$

be the number of supporting neighbours offered through the other triangles.
This is the vertex-to-triangle message. Excluding $a$ prevents counting its
contribution as external support. We may cap $S$ at two when evaluating one
triangle, since its local decision only distinguishes zero, one, and at least
two external supports. For the final root decision we need the total up to
three.

The triangle-to-vertex message
$M_{a\to v}\in\{0,1,2\}$ counts how many of the other two members of $a$ can
survive together **when $v$ is present**, given their external support.
Keeping the whole triangle in this conditional calculation is the essential
change from ordinary edge messages.

## Solve one triangle jointly

Write its members as $v,u_1,u_2$, keep $v$ present, and give $u_i$ an external
support $S_i$ from groups other than this one. Start with both other members
present, then prune them if their external plus internal degree is below three.
The largest surviving state gives:

| External support | Other members that survive | Message to $v$ |
| --- | --- | --- |
| $S_1\ge1$ and $S_2\ge1$ | Both: each has two neighbours inside the triangle | $2$ |
| $S_1=0$ and $S_2\ge2$, or the reverse | Only the member with at least two external neighbours | $1$ |
| All remaining cases | Neither | $0$ |

For example, $(S_1,S_2)=(1,1)$ yields two supports, although neither member
could survive with $v$ alone. Their mutual edge is what lets them stay.
By contrast, $(1,0)$ yields zero: the member with zero external support is
removed, and then the other member has only two neighbours left.

This is a rule for the **ordinary graph 3-core**, counting surviving graph
neighbours. It is not a rule requiring three surviving hyperedges at a vertex,
and it does not delete an entire triangle merely because one member is lost.
A triangle may send one support, so removing all incomplete triangles would
change the problem.

In [ ]:
from numbers import Integral

import numpy as np


def triangle_local_support(s1, s2):
    """Supports supplied by one triangle when its receiving vertex is present.

    s1 and s2 count the other members' neighbours outside this triangle.
    Their mutual edge is retained while deciding whether both can survive.
    """
    for support in (s1, s2):
        if isinstance(support, bool) or not isinstance(support, Integral):
            raise TypeError("External supports must be nonnegative integers.")
        if support < 0:
            raise ValueError("External supports must be nonnegative integers.")
    if s1 >= 1 and s2 >= 1:
        return 2
    if s1 >= 2 or s2 >= 2:
        return 1
    return 0

## Close the probability equations

Let $q_j=\mathbb P(M=j)$ for $j=0,1,2$, and write
$p_j=\mathbb P(S=j)$. The two external branches of a triangle are independent
in the limiting factor tree. Applying the *joint* local rule gives

$$
\boxed{
q_2=(1-p_0)^2,\qquad
q_1=2p_0(1-p_0-p_1),\qquad
q_0=1-q_1-q_2=p_0^2+2p_0p_1.
}
$$

The square in $q_2$ is independence of the two **external branches**, not an
assumption that the two internal triangle edges independently survive.

A vertex receives a Poisson number of independent triangle messages:

$$
S\ \overset{d}=\ \sum_{i=1}^N M_i,
\qquad N\sim\operatorname{Poisson}(\lambda).
$$

Poisson thinning gives an equivalent and useful construction:

$$
S=A+2B,\qquad
A\sim\operatorname{Poisson}(\lambda q_1),\quad
B\sim\operatorname{Poisson}(\lambda q_2),\quad A\perp B.
$$

Its probability generating function is

$$
\mathbb E[z^S]
=\exp\!\left\{\lambda\big[q_1(z-1)+q_2(z^2-1)\big]\right\}.
$$

Taking its first three coefficients yields

$$
\boxed{
\begin{aligned}
p_0&=e^{-\lambda(q_1+q_2)},\\
p_1&=\lambda q_1p_0,\\
p_2&=\left[\lambda q_2+\frac{(\lambda q_1)^2}{2}\right]p_0.
\end{aligned}
}
$$

There are two ways to obtain support two: one message of size two, or two
messages of size one. The displayed equations form a closed two-dimensional
fixed-point problem in $(q_1,q_2)$; $q_0$ is the remaining probability.

## From the messages to the core density

A uniformly chosen root receives messages from all its groups. Its number of
groups is Poisson with the same mean as the excess membership count above, so
its total support has the same distribution $S$. It can remain precisely when
that support is at least three. Therefore

$$
\boxed{
\rho_\triangle(\lambda)
=1-p_0-p_1-p_2
=1-e^{-\lambda(q_1+q_2)}
\left[1+\lambda q_1+\lambda q_2+
                 \frac{(\lambda q_1)^2}{2}\right].
}
$$

The calculation uses messages conditional on the root being present; the final
threshold decides whether this assumption can be sustained by at least three
neighbours. Requiring that root to survive before evaluating its incoming
messages would introduce circular reasoning and lose the pruning construction.

The following functions evaluate the distribution and its update separately
from solving the fixed point. All probabilities refer to a specified message
law; the density becomes a core prediction only after choosing the physical
fixed point.

In [ ]:
def _triangle_inputs(q, lam):
    """Validate (q1, q2), with the unused probability q0 inferred from their sum."""
    q = np.asarray(q, dtype=float)
    lam = float(lam)
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("lambda must be finite and nonnegative.")
    if q.shape != (2,) or not np.all(np.isfinite(q)):
        raise ValueError("q must contain the two finite probabilities (q1, q2).")
    if np.any(q < 0) or np.sum(q) > 1 + 8 * np.finfo(float).eps:
        raise ValueError("q1 and q2 must be nonnegative and sum to at most one.")
    return q, lam


def triangle_support_probabilities(q, lam):
    """Return P(S=0), P(S=1), P(S=2) for compound Poisson support S=A+2B."""
    q, lam = _triangle_inputs(q, lam)
    rate1, rate2 = lam * q
    p0 = np.exp(-lam * np.sum(q))
    if p0 == 0:
        # At extremely large rates these probabilities underflow to zero.
        return np.zeros(3)
    return p0 * np.array([1.0, rate1, rate2 + rate1**2 / 2])


def triangle_messages(q, lam):
    """One update of the joint triangle-message law, returned as (q1, q2)."""
    q, lam = _triangle_inputs(q, lam)
    p0, p1, _ = triangle_support_probabilities(q, lam)
    positive_support = -np.expm1(-lam * np.sum(q))
    at_least_two = max(0.0, positive_support - p1)
    return np.array([2 * p0 * at_least_two, positive_support**2])


def triangle_density(q, lam):
    """Probability of at least three supporting graph neighbours for a given q."""
    q, lam = _triangle_inputs(q, lam)
    _, p1, p2 = triangle_support_probabilities(q, lam)
    result = -np.expm1(-lam * np.sum(q)) - p1 - p2
    return float(np.clip(result, 0.0, 1.0))


def triangle_jacobian(q, lam):
    """Derivative of the two-dimensional probability update with respect to q."""
    q, lam = _triangle_inputs(q, lam)
    p0 = np.exp(-lam * np.sum(q))
    common = 2 * lam * p0
    return common * np.array(
        [
            [p0 - 1 + 2 * lam * q[0] * p0, 2 * p0 - 1 + 2 * lam * q[0] * p0],
            [1 - p0, 1 - p0],
        ]
    )

## Which fixed point represents pruning?

Zero messages are always a fixed point. Starting an iteration at zero would
therefore predict an empty core even when another solution exists. Pruning
instead starts with every possible support available: every triangle message has value two, so $(q_1,q_2)=(0,1)$ initially.

The local pruning rule is monotone in its external supports. Each update from
this initial state can only remove support in **stochastic order**. In terms of
probabilities, the decreasing quantities are the two tails

$$
\mathbb P(M\ge1)=q_1+q_2,\qquad
\mathbb P(M\ge2)=q_2.
$$

The probability $q_1$ alone need not decrease: a triangle can lose one of two
supporting members and start sending one support. Coordinatewise comparisons
of $(q_1,q_2)$ would consequently use the wrong ordering.

Iteration from the maximal message law dominates any fixed point and converges
to the maximal physical law. On the factor tree this has the interpretation of
pruning successively deeper finite neighbourhoods with fully supporting
boundary conditions. It selects the appropriate infinite-tree limit.

The simple implementation below is an independent check of the solver used
later. Near the transition it can converge very slowly. A small change between
iterations is a residual, not a guarantee of a comparably small error in the
solution when the update is nearly tangent to the identity. The return value
includes `converged` so a stopped iteration is never silently called a solution.

In [ ]:
def triangle_iteration(lam, tolerance=1e-12, max_iterations=100000):
    """Iterate from all messages equal to two; report convergence diagnostics.

    The stochastic tails q1+q2 and q2 decrease, although q1 may increase.
    This diagnostic is deliberately independent of the scalar branch solver.
    """
    q, lam = _triangle_inputs([0.0, 1.0], lam)
    if not np.isfinite(tolerance) or tolerance <= 0:
        raise ValueError("tolerance must be finite and positive.")
    if isinstance(max_iterations, bool) or not isinstance(max_iterations, Integral):
        raise TypeError("max_iterations must be a positive integer.")
    if max_iterations < 1:
        raise ValueError("max_iterations must be a positive integer.")
    for iteration in range(1, max_iterations + 1):
        updated = triangle_messages(q, lam)
        q = updated
        residual = float(np.max(np.abs(triangle_messages(q, lam) - q)))
        if residual <= tolerance:
            break
    return {
        "lambda": lam,
        "q1": float(q[0]),
        "q2": float(q[1]),
        "rho": triangle_density(q, lam),
        "residual": residual,
        "iterations": iteration,
        "converged": residual <= tolerance,
    }

## Reduce the fixed points without discarding the joint messages

We can parameterize every positive solution by its probability of no external
support. This is an algebraic reduction of the joint triangle system, not an
assumption that two edges in a triangle are independent.

Set $a=p_0$ and $z=-\log a>0$, so $a=e^{-z}$. At a fixed point,

$$
q_2=(1-a)^2,
\qquad
q_1=2a(1-a-\lambda q_1a)
=\frac{2a(1-a)}{1+2\lambda a^2}.
$$

The equation defining $p_0$ is therefore

$$
H(z,\lambda)
=\lambda(1-a)^2+
 \frac{2\lambda a(1-a)}{1+2\lambda a^2}-z=0,
\qquad a=e^{-z}.
$$

For fixed $z>0$, its derivative in $\lambda$ is strictly positive:

$$
\partial_\lambda H
=(1-a)^2+\frac{2a(1-a)}{(1+2\lambda a^2)^2}>0.
$$

There is exactly one positive $\lambda$ for that $z$. Multiplying by the
positive denominator gives a quadratic,

$$
2a^2(1-a)^2\lambda^2+
\underbrace{(1-a^2-2a^2z)}_{B(z)}\lambda-z=0.
$$

Its positive root, written to avoid subtracting nearly equal numbers, is

$$
\boxed{
\Lambda(z)=
\frac{2z}{B(z)+\sqrt{B(z)^2+8a^2(1-a)^2z}},
\qquad a=e^{-z}.
}
$$

Every positive fixed point is on this curve, and each point on the curve
satisfies the original two-dimensional equations. The zero solution is kept
separately. Larger $z$ means smaller $p_0$, larger $q_2=(1-p_0)^2$, and larger
$q_1+q_2=z/\lambda$ at fixed $\lambda$. Thus the **largest $z$ root** is exactly
the maximal solution in both stochastic tails.

In [ ]:
def _triangle_bisect(function, left, right, tolerance=2e-15):
    """Find a scalar zero in a sign bracket, without a grid-resolution error."""
    left_value = function(left)
    right_value = function(right)
    if left_value == 0:
        return float(left)
    if right_value == 0:
        return float(right)
    if left_value * right_value > 0:
        raise ValueError("Bisection needs opposite signs at the interval ends.")
    for _ in range(160):
        middle = left + (right - left) / 2
        middle_value = function(middle)
        if middle_value == 0 or right - left <= tolerance * max(1, abs(middle)):
            return float(middle)
        if left_value * middle_value > 0:
            left, left_value = middle, middle_value
        else:
            right = middle
    raise RuntimeError("Bisection did not reach the requested accuracy.")


def _triangle_lambda_for_z(z):
    """Parameterize the positive fixed-point branch by z=-log P(S=0)."""
    z = float(z)
    if not np.isfinite(z) or z <= 0:
        raise ValueError("A positive fixed-point branch requires finite z > 0.")
    a = np.exp(-z)
    positive = -np.expm1(-z)
    # B=1-exp(-2z)*(1+2z). Its short series avoids cancellation near zero.
    if z < 1e-4:
        b = z**2 * (2 + z * (-8 / 3 + z * (2 + z * (-16 / 15 + z * 4 / 9))))
    else:
        b = -np.expm1(-2 * z) - 2 * z * a**2
    discriminant = np.hypot(b, np.sqrt(8 * z) * a * positive)
    return float(2 * z / (b + discriminant))


def _triangle_scalar_residual(z, lam):
    """H(z, lambda); its positive zero is equivalent to the joint fixed point."""
    a = np.exp(-z)
    positive = -np.expm1(-z)
    q1 = 2 * a * positive / (1 + 2 * lam * a**2)
    return float(lam * (q1 + positive**2) - z)


def _triangle_scalar_derivative(z, lam):
    """Partial derivative H_z with lambda held fixed; zero at the fold."""
    a = np.exp(-z)
    positive = -np.expm1(-z)
    denominator = 1 + 2 * lam * a**2
    return float(
        2 * lam * a * positive
        + 2 * lam * a * (2 * a - 1 + 2 * lam * a**2) / denominator**2
        - 1
    )


def _triangle_prediction_at_z(z, lam):
    """Recover the two message probabilities and their checked density."""
    a = np.exp(-z)
    positive = -np.expm1(-z)
    q = np.array([2 * a * positive / (1 + 2 * lam * a**2), positive**2])
    return {
        "lambda": float(lam),
        "z": float(z),
        "p0": float(a),
        "q1": float(q[0]),
        "q2": float(q[1]),
        "rho": triangle_density(q, lam),
        "residual": float(np.max(np.abs(triangle_messages(q, lam) - q))),
    }


def triangle_branch(z):
    """Return a positive branch point, including the lower, unstable branch.

    This is useful for the fixed-point diagram; it does not choose the
    physically maximal solution at a specified lambda.
    """
    lam = _triangle_lambda_for_z(z)
    return _triangle_prediction_at_z(z, lam)

## Locate the first positive solution and determine its transition type

Near $z=0$ the branch satisfies
$\Lambda(z)\sim(2z)^{-1/2}$; for $z\to\infty$ it satisfies
$\Lambda(z)\sim z$. Both ends therefore require an unbounded parameter.
The first positive solution occurs at an interior minimum of $\Lambda$.

There is no continuous bifurcation from zero at any finite $\lambda$. The
Jacobian of the original map at zero is

$$
D\mathcal T_\lambda(0,0)=
\begin{pmatrix}0&2\lambda\\0&0\end{pmatrix},
$$

whose two eigenvalues are zero. In particular, the zero fixed point is locally
stable for every finite parameter, and the identity minus this Jacobian is
invertible. A positive branch cannot merge continuously with it at finite
$\lambda$. The first nonzero branch must have a positive jump in density.

Numerically the positive branch has one fold. Below its minimum there are no
positive fixed points; above it there are two: a smaller, unstable solution
and the larger, pruning-selected solution. Notebook 10 plots the whole branch,
checks both sides, and compares with direct iteration from all messages active.
We use this numerically checked branch structure when bracketing the maximal
solution; we do not claim a separate analytic uniqueness theorem for the fold.

Implicit differentiation gives
$\Lambda'(z)=-H_z/H_\lambda$, so the minimum can be refined by solving
$H_z(z,\Lambda(z))=0$. The explicit derivative is

$$
H_z=2\lambda a(1-a)+
\frac{2\lambda a(2a-1+2\lambda a^2)}{(1+2\lambda a^2)^2}-1.
$$

The initial grid in the code below brackets the minimum; bisection on the
continuous derivative determines its precision. This avoids treating a grid
point as the threshold or missing two closely spaced roots near the fold.
The returned Jacobian spectral radius provides a second tangency diagnostic:
at the fold the two-dimensional update has an eigenvalue equal to one.

The fold is nondegenerate: the numerical derivatives at the critical solution
are $H_\lambda\approx0.34663>0$ and $H_{zz}\approx-0.70217<0$.
The latter is independently checked by central differences of the displayed
analytic $H_z$, using several step sizes. A Taylor expansion consequently gives

$$
0\approx H_\lambda(\lambda-\lambda_c)
       +\frac12H_{zz}(z-z_c)^2,
\qquad
z-z_c\approx\pm
\sqrt{-\frac{2H_\lambda}{H_{zz}}(\lambda-\lambda_c)}.
$$

This is the saddle-node mechanism: two positive solutions are born together
above the threshold, and pruning selects the plus sign. It explains both the
positive jump from the separate zero branch and the steep square-root departure
from the critical positive solution. A steep curve alone would not establish a
jump; the nonzero critical density and the absence of a finite-parameter
bifurcation from zero are the relevant evidence here.


In [ ]:
def triangle_critical_point():
    """Locate the fold and return its positive branch limit and diagnostics."""
    parameters = np.geomspace(1e-4, 20.0, 257)
    lambdas = np.array([_triangle_lambda_for_z(z) for z in parameters])
    minimum = int(np.argmin(lambdas))
    if minimum in (0, len(parameters) - 1):
        raise RuntimeError("The positive-branch minimum was not bracketed.")

    def tangent_residual(z):
        return _triangle_scalar_derivative(z, _triangle_lambda_for_z(z))

    z = _triangle_bisect(
        tangent_residual, parameters[minimum - 1], parameters[minimum + 1]
    )
    result = triangle_branch(z)
    q = [result["q1"], result["q2"]]
    result["tangent_residual"] = abs(tangent_residual(z))
    result["spectral_radius"] = float(
        np.max(np.abs(np.linalg.eigvals(triangle_jacobian(q, result["lambda"]))))
    )
    return result


def triangle_prediction(lam, critical=None):
    """Select the largest fixed point, using the critical right-hand convention."""
    _, lam = _triangle_inputs([0.0, 0.0], lam)
    if critical is None:
        critical = triangle_critical_point()
    if lam < critical["lambda"]:
        return {
            "lambda": lam,
            "z": 0.0,
            "p0": 1.0,
            "q1": 0.0,
            "q2": 0.0,
            "rho": 0.0,
            "residual": 0.0,
        }
    # At a double root, differences of a few floating-point units do not
    # resolve the split. Return the explicitly labelled right-hand limit.
    if lam - critical["lambda"] <= 8 * np.spacing(critical["lambda"]):
        z = critical["z"]
    else:
        # H is positive at z_c above the fold, and nonpositive for z >= lambda
        # since q1+q2 <= 1. This brackets the largest positive root.
        z = _triangle_bisect(
            lambda value: _triangle_scalar_residual(value, lam),
            critical["z"],
            max(lam, critical["z"] + 1),
        )
    return _triangle_prediction_at_z(z, lam)

## Translate the message fold into a density expansion

The square-root change in the messages does not yet give the change in the
fraction of vertices that remain. We must also differentiate the density
formula. In this section write $u=e^{-z}$ for the no-support probability,
so that the density coefficient $a_\triangle$ is not confused with the
earlier abbreviation $a=p_0$.

For fixed $\lambda$, define

$$
d=1+2\lambda u^2,\qquad B=2u-1+2\lambda u^2,\qquad
q_1=\frac{2u(1-u)}d,\qquad q_2=(1-u)^2.
$$

Since $u_z=-u$, the message derivatives are

$$
\begin{aligned}
q_{1,z}&=\frac{2uB}{d^2},&
q_{2,z}&=2u(1-u),\\
q_{1,zz}&=-\frac{2u(4u-1+6\lambda u^2)}{d^2}
           +\frac{16\lambda u^3B}{d^3},&
q_{2,zz}&=2u(2u-1).
\end{aligned}
$$

Consequently the derivatives of the scalar residual are

$$
H_\lambda=(1-u)^2+\frac{2u(1-u)}{d^2}>0,\qquad
H_{zz}=\lambda(q_{1,zz}+q_{2,zz}).
$$

To keep the density defined even a little off the fixed-point curve, insert
these $q_1(z,\lambda),q_2(z,\lambda)$ into the original support distribution:

$$
\varrho(z,\lambda)
=1-e^{-\lambda(q_1+q_2)}
 \left[1+\lambda(q_1+q_2)+\frac{(\lambda q_1)^2}{2}\right].
$$

On a fixed point this is the physical density $\rho_\triangle(\lambda)$.
Off the curve, $e^{-\lambda(q_1+q_2)}$ need not equal $u$. Retaining the
original exponential makes numerical differentiation unambiguous.
Differentiating at fixed $\lambda$ gives

$$
\varrho_z=e^{-\lambda(q_1+q_2)}
\left[
\lambda(q_{1,z}+q_{2,z})
 \left\{\lambda(q_1+q_2)+\frac{(\lambda q_1)^2}{2}\right\}
-\lambda^2q_1q_{1,z}
\right].
$$

All derivatives below are evaluated at $(z_c,\lambda_c)$. Write
$\delta=\lambda-\lambda_c>0$ and $\eta=z-z_c$.
Because $H=H_z=0$, Taylor expansion yields

$$
0=H_\lambda\delta+\tfrac12H_{zz}\eta^2
  +O(\delta^2+|\eta|\delta+|\eta|^3).
$$

Here $H_\lambda>0$ and $H_{zz}<0$, so the upper branch has

$$
\eta=\beta_\triangle\sqrt{\delta}+O(\delta),
\qquad
\beta_\triangle=\sqrt{-\frac{2H_\lambda}{H_{zz}}}>0.
$$

Expanding the smooth density map next gives
$\varrho-\rho_c=\varrho_z\eta+O(\delta)$, since $\eta^2=O(\delta)$.
Therefore

$$
\boxed{
\rho_\triangle(\lambda)
=\rho_{c,\triangle}
 +a_\triangle\sqrt{\lambda-\lambda_{c,\triangle}}
 +O(\lambda-\lambda_{c,\triangle}),\qquad
a_\triangle=\varrho_z\sqrt{-\frac{2H_\lambda}{H_{zz}}}.
}
$$

The nonzero derivative $\varrho_z>0$ means that the message fold is visible
in the measured density. Both the positive critical density and this local
expansion are consequences of the cavity equations. They do not prove a
terminal-core law or a finite-size fluctuation theorem for the random graphs.


In [ ]:
def _triangle_fold_derivatives(z, lam):
    """Differentiate H and the full density after eliminating q1 and q2."""
    z, lam = float(z), float(lam)
    if not np.isfinite(z) or z <= 0 or not np.isfinite(lam) or lam < 0:
        raise ValueError("Use finite z > 0 and finite lambda >= 0.")
    u = np.exp(-z)
    positive = -np.expm1(-z)
    denominator = 1 + 2 * lam * u**2
    numerator = 2 * u - 1 + 2 * lam * u**2
    q1 = 2 * u * positive / denominator
    q2 = positive**2
    q1_z = 2 * u * numerator / denominator**2
    q2_z = 2 * u * positive
    q1_zz = (
        -2 * u * (4 * u - 1 + 6 * lam * u**2) / denominator**2
        + 16 * lam * u**3 * numerator / denominator**3
    )
    q2_zz = 2 * u * (2 * u - 1)
    total_rate = lam * (q1 + q2)
    rho_z = np.exp(-total_rate) * (
        lam * (q1_z + q2_z) * (total_rate + (lam * q1) ** 2 / 2) - lam**2 * q1 * q1_z
    )
    return {
        "H_lambda": float(q2 + 2 * u * positive / denominator**2),
        "H_zz": float(lam * (q1_zz + q2_zz)),
        "rho_z": float(rho_z),
    }


def triangle_fold(critical=None):
    """Return the density square-root coefficient and its fold derivatives."""
    if critical is None:
        critical = triangle_critical_point()
    result = dict(critical)
    z, lam = result["z"], result["lambda"]
    if (
        max(
            abs(_triangle_scalar_residual(z, lam)),
            abs(_triangle_scalar_derivative(z, lam)),
        )
        > 1e-8
    ):
        raise ValueError("The supplied critical point must satisfy H=H_z=0.")
    derivatives = _triangle_fold_derivatives(z, lam)
    if not (
        derivatives["H_lambda"] > 0
        and derivatives["H_zz"] < 0
        and derivatives["rho_z"] > 0
    ):
        raise RuntimeError("The positive, nondegenerate density fold was not verified.")
    message_amplitude = np.sqrt(-2 * derivatives["H_lambda"] / derivatives["H_zz"])
    result.update(derivatives)
    result["message_amplitude"] = float(message_amplitude)
    result["a"] = float(derivatives["rho_z"] * message_amplitude)
    return result

## Check the derivatives and the square-root prediction independently

The coefficient is calculated from derivatives, rather than fitted to core
densities. Three checks help catch a mistaken derivative or a wrong branch:

1. Compare $H_\lambda$ with a central difference of $H$ in $\lambda$.
   Compare $H_{zz}$ both with a central difference of $H_z$ and with the
   second difference of $H$ itself. Vary the step size.
2. Compare $\varrho_z$ with a central difference of the original density
   function after substituting $q_1(z,\lambda),q_2(z,\lambda)$.
3. Solve the original scalar equation on the maximal branch at
   $\lambda_c+\delta$ and calculate
   $(\rho_\triangle(\lambda_c+\delta)-\rho_c)/\sqrt{\delta}$.
   This ratio should approach $a_\triangle$; the remaining density error
   should be of order $\delta$.

The function below returns these deterministic checks without saving files.
For example, run the following in a separate analysis notebook or console:

~~~python
fold = triangle_fold()
checks = triangle_fold_checks(fold)
fold
checks["derivatives"]
checks["branch"]
~~~

The verified values, rounded for reading, are:

| Quantity | Value |
| --- | ---: |
| $\lambda_{c,\triangle}$ | 1.22859824 |
| $\rho_{c,\triangle}$ | 0.08111206 |
| $H_\lambda$ | 0.34663025 |
| $H_{zz}$ | -0.70217159 |
| $\varrho_z$ | 0.28033751 |
| $\beta_\triangle$ | 0.99363435 |
| $a_\triangle$ | 0.27855298 |

For $\delta=10^{-4},10^{-6},10^{-8}$ the direct-branch density ratios are
approximately $0.28221843$, $0.27891839$, and $0.27858951$.
Their approach to $a_\triangle$ verifies the leading term; it does not
make the square-root curve exact away from the fold.

Using an extremely small difference step can increase cancellation error.
Likewise, taking $\delta$ down to the uncertainty of the floating-point
critical parameter no longer resolves two separate roots reliably.
The checks use several moderate steps and stop the branch comparison at
$10^{-8}$, rather than interpreting roundoff as a physical correction.
Loading this notebook still defines functions only: the optional calls above
perform no random-graph simulation.


In [ ]:
def triangle_fold_checks(critical=None):
    """Compare analytic fold derivatives with differences and direct roots."""
    fold = triangle_fold(critical)
    z, lam = fold["z"], fold["lambda"]
    derivative_rows = []
    for step in (1e-3, 3e-4, 1e-4, 3e-5, 1e-5):
        h_lambda = (
            _triangle_scalar_residual(z, lam + step)
            - _triangle_scalar_residual(z, lam - step)
        ) / (2 * step)
        h_zz = (
            _triangle_scalar_derivative(z + step, lam)
            - _triangle_scalar_derivative(z - step, lam)
        ) / (2 * step)
        h_zz_residual = (
            _triangle_scalar_residual(z + step, lam)
            - 2 * _triangle_scalar_residual(z, lam)
            + _triangle_scalar_residual(z - step, lam)
        ) / step**2
        rho_z = (
            _triangle_prediction_at_z(z + step, lam)["rho"]
            - _triangle_prediction_at_z(z - step, lam)["rho"]
        ) / (2 * step)
        approximations = {
            "H_lambda": h_lambda,
            "H_zz": h_zz,
            "rho_z": rho_z,
        }
        derivative_rows.append(
            {
                "step": step,
                **{key: float(value) for key, value in approximations.items()},
                **{
                    key + "_error": float(value - fold[key])
                    for key, value in approximations.items()
                },
                "H_zz_from_residual": float(h_zz_residual),
                "H_zz_from_residual_error": float(h_zz_residual - fold["H_zz"]),
            }
        )

    branch_rows = []
    for delta in (1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8):
        prediction = triangle_prediction(lam + delta, critical=fold)
        ratio = (prediction["rho"] - fold["rho"]) / np.sqrt(delta)
        branch_rows.append(
            {
                "delta": delta,
                "rho": prediction["rho"],
                "density_ratio": float(ratio),
                "relative_amplitude_error": float(ratio / fold["a"] - 1),
                "remainder_over_delta": float(
                    (prediction["rho"] - fold["rho"] - fold["a"] * np.sqrt(delta))
                    / delta
                ),
                "fixed_point_residual": prediction["residual"],
            }
        )
    return {"fold": fold, "derivatives": derivative_rows, "branch": branch_rows}

## Numerical prediction and what its precision means

Solving the fold equations gives approximately

$$
\begin{aligned}
\lambda_{c,\triangle}&=1.22859824, &
z_c&=0.57467124, &
p_{0,c}&=0.56288989,\\
q_{1,c}&=0.27668021, &
q_{2,c}&=0.19106525, &
\rho_{c,\triangle}&=0.08111206.
\end{aligned}
$$

The predicted mean graph degree at onset is $2\lambda_c\approx2.45720$.
The jump is about **8.11% of the vertices**. Below the threshold the density
prediction is zero. At exactly the threshold `triangle_prediction` returns
the positive branch's right-hand limit, for the same plotting convention used
for the CM curve in notebook 05. This is not a deterministic finite-$n$
prediction at that parameter.

For a curve, compute the critical point once and reuse it:

```python
critical = triangle_critical_point()
curve = [triangle_prediction(lam, critical) for lam in lambdas]
```

Machine-level residuals assess the equations' numerical solution. They do not
imply equally precise physical thresholds for finite graphs or establish the
cavity assumption as a theorem. Near a double root, the message value is more
sensitive than the critical parameter: roundoff and a tiny change in $\lambda$
can create a square-root change in the solution. The eight decimal places
above are useful for reproducible numerical checks, while plots and scientific
comparisons should use fewer significant figures.

## Independent checks and useful limiting cases

Notebook 10 and the automated tests check the calculations in several ways.

- **One triangle:** exhaustively prune the two other members for small integer
  external supports and compare the surviving count with `triangle_local_support`.
  In particular, zero external support leaves an isolated triangle with no core.
- **Probability distribution:** convolve independent Poisson variables $A$ and
  $B$ to recover the support probabilities and density, independently of the
  generating-function implementation.
- **Critical point:** compare the derivative-based answer with direct
  minimization of the explicit branch curve. Check $H=0$, $H_z=0$, the
  original two message residuals, and the Jacobian eigenvalue one.
- **Root selection:** compare the scalar largest root against independent
  two-dimensional iteration from all messages equal to two, including values
  on either side of the fold. A very small iteration cap must report failure
  to converge rather than invent a solution.
- **Special cases:** at $\lambda=0$ there is no support and the predicted core
  is empty. At large $\lambda$, $q_2$ and the density approach one. The zero
  message law remains a fixed point throughout, illustrating why initialization
  matters.

The diagnostic figures show the stable and unstable branches and the slow
approach of iteration near the fold. A smoothed average over finite graphs is
not a substitute for these checks: a mixture of near-empty and substantial
cores can produce a mean that looks continuous even when the limiting
prediction jumps.

## Why this remains a cavity prediction for our ensemble

The local triangle calculation is exact when its three vertices are distinct
and the external branches do not reconnect. Independence is exact on the
limiting factor tree for any fixed number of update rounds. The assumptions
enter when we use its infinite-round message limit to predict the fraction
remaining after full pruning of a large finite graph.

Local weak convergence alone does not justify exchanging the limits
$n\to\infty$ and the number of pruning rounds $t\to\infty$. Near the fold,
pruning can take many rounds and a small perturbation can trigger a large
cascade. A proof would need control of that late pruning process, along with
concentration and an appropriate stability argument away from criticality.
We do not supply such a theorem here.

Our finite simple projection also merges repeated neighbours. Triangle-slot
messages would overcount them if applied literally to every finite sample.
For example, two identical communities on the same three individuals produce
only one simple triangle, whose 3-core is empty, even though counting each
group's neighbours separately would give spurious support. Such coincidences
vanish from a typical fixed-radius neighbourhood as $n$ grows, but their sparse
number alone does not prove they cannot initiate critical pruning cascades.
The simulations therefore use **direct ordinary graph pruning**, rather than
silently replacing it with an incidence-counting algorithm.

The scientific validation in notebook 10 compares this prediction with those
simulations across sizes, treats the critical window separately, and inspects
individual outcomes. Agreement is evidence for the cavity description; it
does not turn it into a rigorous finite-graph result.

## Relation to the assignment and literature

The local rule and fixed-point closure above complete Section 2 of the
[project assignment](../3-core-random-intersection-graphs.pdf). The scalar
branch parameterization, derivative-based fold solver, and numerical
cross-checks are our calculations for its pure-triangle specialization.

- [Janson and Łuczak (2007), *A simple solution to the k-core problem*](https://arxiv.org/pdf/math/0508453)
  provide the configuration-model theorem specialized in notebook 05. Its
  ordinary edge-based recursion does not directly apply to triangle communities.
- [van der Hofstad, Komjáthy and Vadon (2021), *Random intersection graphs with communities*](https://arxiv.org/pdf/1809.02514)
  provide the local-limit and limiting-degree framework used here.
- Their [*Phase transition in random intersection graphs with communities*](https://arxiv.org/pdf/1905.06253)
  treats giant connected components and percolation, rather than proving this
  3-core prediction.
- [Gleeson and Melnik (2009), *Analytical results for bond percolation and k-core sizes on clustered networks*](https://arxiv.org/pdf/0811.4511)
  is a precedent for retaining clique structure in a pruning calculation. Its
  generalized Trapman ensemble differs from our overlapping-triangle model;
  its formulas are not being quoted as an exact solution of our ensemble.

Matching the triangle and CM limiting degree laws leaves different motif and
neighbour-support correlations. Notebook 10 compares their thresholds, jump
sizes, and densities without attributing every difference to one clustering
coefficient alone.

The combined table, density curves, and independent derivative checks are
executed in [the deterministic comparison notebook](theory_comparison.ipynb).
Use `make theory-figures` to regenerate them for the paper. The stochastic
conditional CDF and quantile helpers in notebook 05 accept the triangle
coefficient too; using them for triangles is a testable approximation, not a
consequence of the configuration-model fluctuation theorem.